# WiDS Datathon 2024 Challenge 1
## 유방암 전이 진단 90일 이내 예측

**대회**: WiDS Datathon 2024 — Equity in Healthcare  
**목표**: 유방암 환자가 전이성 암 진단을 받기까지의 기간이 90일 이내인지 이진 분류  
**타겟**: `DiagPeriodL90D` (1 = 90일 이내, 0 = 90일 초과)

### 핵심 전략
- **특성 공학**: ICD 코드 버전 분류, 종양 위치 추출, 나이브 베이즈 기반 인종 결측치 보완
- **모델링**: CatBoost + RandomForest + XGBoost + LightGBM 스태킹 앙상블
- **검증**: 10-Fold Stratified K-Fold Cross Validation

## 1. 라이브러리 불러오기

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'  # 한글 폰트 (Windows)
plt.rcParams['axes.unicode_minus'] = False

## 2. 데이터 불러오기

In [ ]:
DATA_DIR = './data'

df = pd.read_csv(f'{DATA_DIR}/training.csv')
tdf = pd.read_csv(f'{DATA_DIR}/test.csv')
ss = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train shape: {df.shape}')
print(f'Test shape:  {tdf.shape}')
print(f'Sample submission: {ss.shape}')

In [ ]:
target = 'DiagPeriodL90D'

# 클래스 분포 확인
class_counts = df[target].value_counts()
plt.figure(figsize=(6, 4))
class_counts.plot(kind='bar')
plt.title('Target Class Distribution (DiagPeriodL90D)')
plt.xlabel('Class (0: >90일, 1: ≤90일)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(class_counts)

In [ ]:
df.info()

## 3. 특성 공학 (Feature Engineering)

### 3-1. N02, PM25, commute_time 결측치 보완
- `patient_zip3`의 앞 2자리(`patient_zip2`)로 그룹 구성
- 동일 zip2 그룹의 평균으로 결측치 대체

In [ ]:
df['patient_zip2'] = df['patient_zip3'].astype(str).str[:2]
tdf['patient_zip2'] = tdf['patient_zip3'].astype(str).str[:2]

target_cols = ['N02', 'PM25', 'commute_time']

for dataset in [df, tdf]:
    for col in target_cols:
        dataset[col] = dataset.groupby('patient_zip2')[col].transform(
            lambda x: x.fillna(x.mean())
        )

for col in target_cols:
    print(f'{col} 결측치 수: {df[col].isnull().sum()}')

### 3-2. 환경 오염 복합 지표 생성
- N02 × PM25 × commute_time (MinMax 스케일링 후 곱)
- 대기오염 + 통근시간의 복합적 건강 영향 반영

In [ ]:
scaler = MinMaxScaler()

df_scaled = pd.DataFrame(
    scaler.fit_transform(df[target_cols]),
    columns=target_cols,
    index=df.index
)
df['N02xPM25xcommute_time'] = df_scaled['N02'] * df_scaled['PM25'] * df_scaled['commute_time']

tdf_scaled = pd.DataFrame(
    scaler.transform(tdf[target_cols]),
    columns=target_cols,
    index=tdf.index
)
tdf['N02xPM25xcommute_time'] = tdf_scaled['N02'] * tdf_scaled['PM25'] * tdf_scaled['commute_time']

print('N02xPM25xcommute_time 피처 생성 완료')

### 3-3. Train + Test 병합 (일관된 전처리)

In [ ]:
cat_cols = list(tdf.columns[tdf.dtypes == 'object'])
tdf[target] = np.nan

df = pd.concat([df, tdf[df.columns]], axis=0).reset_index(drop=True)
print(f'병합 후 shape: {df.shape}')

### 3-4. 전이암 코드 관련 파생 피처
- `clust`: 전이암 코드 길이가 4자리 → 세분화 여부
- `is_female`: 진단 설명에 'female' 포함 여부
- `meta_code4`: 전이암 코드 앞 4자리 (부위 그룹화)

In [ ]:
df['clust'] = (df['metastatic_cancer_diagnosis_code'].str.len() == 4).astype('int')
df['is_female'] = df['breast_cancer_diagnosis_desc'].str.contains('female', na=False).astype('int')

df['meta_code4'] = df['metastatic_cancer_diagnosis_code'].astype(str).apply(
    lambda x: x[:4] if len(x) >= 5 else x
)
df = df.drop(columns=['metastatic_cancer_diagnosis_code'])

print('clust 분포:')
print(df['clust'].value_counts())
print('\nis_female 분포:')
print(df['is_female'].value_counts())
print(f'\nmeta_code4 고유값 수: {df["meta_code4"].nunique()}')

### 3-5. ICD 코드 버전 분류
- ICD-10: C로 시작 / ICD-9: 그 외

In [ ]:
df['ICD_version'] = np.where(
    df['breast_cancer_diagnosis_code'].astype(str).str.startswith('C'),
    'ICD-10',
    'ICD-9'
)
print(df['ICD_version'].value_counts())

### 3-6. 종양 위치 추출 (tumor_site, tumor_site2)
- 진단 설명(desc)에서 종양 위치 키워드 추출
- 최종 모델에는 `tumor_site` (5개 범주) 사용

In [ ]:
df['desc_lower'] = df['breast_cancer_diagnosis_desc'].str.lower()

# tumor_site: 5개 범주
df['tumor_site'] = np.select(
    [
        df['desc_lower'].str.contains('inner', na=False),
        df['desc_lower'].str.contains('outer', na=False),
        df['desc_lower'].str.contains('central', na=False),
        df['desc_lower'].str.contains('overlapping', na=False),
    ],
    ['Inner', 'Outer', 'Central', 'Overlapping'],
    default='Unspecified'
)

# tumor_site2: 8개 세부 범주 (인종 결측치 보완에 활용)
df['tumor_site2'] = np.select(
    [
        df['desc_lower'].str.contains('upper-inner', na=False),
        df['desc_lower'].str.contains('lower-inner', na=False),
        df['desc_lower'].str.contains('upper-outer', na=False),
        df['desc_lower'].str.contains('lower-outer', na=False),
        df['desc_lower'].str.contains('axillary', na=False),
        df['desc_lower'].str.contains('central', na=False),
        df['desc_lower'].str.contains('overlapping', na=False),
    ],
    ['Upper-inner', 'Lower-inner', 'Upper-outer', 'Lower-outer',
     'Axillary tail', 'Central', 'Overlapping'],
    default='Unspecified'
)

print('tumor_site 분포:')
print(df['tumor_site'].value_counts())

### 3-7. 인종(patient_race) 결측치 보완 — Naive Bayes

**전략**: P(race | tumor_site2, zip3) ∝ P(tumor_site2 | race) × P(zip3 | race) × P(race)
- P(tumor_site2 | race): 논문 기반 인종별 종양 위치 확률
- P(zip3 | race): 데이터에서 계산한 지역별 인종 분포
- 사후 확률 ≥ 0.8일 때만 대체 (불확실한 추론 배제)

In [ ]:
races = ['White', 'Black', 'Asian', 'Hispanic', 'Other']

# P(tumor_site2 | race) — 논문 기반 확률 (%)
paper_data = {
    'Central':       [ 8.5,  7.7, 10.3,  8.9, 16.79],
    'Upper-inner':   [16.6, 17.0, 19.7, 17.7, 13.87],
    'Lower-inner':   [ 8.3,  9.8,  8.3,  8.2,  6.57],
    'Upper-outer':   [54.9, 53.4, 50.6, 53.6, 51.46],
    'Lower-outer':   [10.7, 10.8, 10.4, 10.7, 10.58],
    'Axillary tail': [ 1.0,  1.3,  0.7,  0.8,  0.73],
    'Unspecified':   [55.72, 55.88, 64.62, 62.61, 49.93],
    'Overlapping':   [ 9.43, 10.53, 10.03,  7.48,  9.84]
}
tumor_site_prob_per_race = pd.DataFrame(paper_data, index=races).T

# P(zip3 | race)
df_valid = df[df['patient_race'].notna()].copy()
zip3_race_counts = df_valid.groupby(['patient_race', 'patient_zip3']).size().unstack(fill_value=0)
zip3_given_race_prob = zip3_race_counts.div(zip3_race_counts.sum(axis=1), axis=0)

# P(race)
race_prob = df['patient_race'].value_counts(normalize=True).reindex(races, fill_value=0)

print('P(race) 사전 확률:')
print(race_prob)

In [ ]:
missing = df[df['patient_race'].isna()].copy()
total_missing = len(missing)
filled_count = 0
predicted_labels = {}
threshold = 0.8

for idx, row in missing.iterrows():
    tumor = row['tumor_site2']
    zip3 = row['patient_zip3']

    probs = {}
    for race in races:
        p_tumor = tumor_site_prob_per_race.at[tumor, race] / 100 if tumor in tumor_site_prob_per_race.index else 0
        p_zip = zip3_given_race_prob.at[race, zip3] if zip3 in zip3_given_race_prob.columns else 0
        p_race = race_prob[race]
        probs[race] = p_tumor * p_zip * p_race

    total_prob = sum(probs.values())
    if total_prob == 0:
        continue

    normalized_probs = {k: v / total_prob for k, v in probs.items()}
    best_race = max(normalized_probs, key=normalized_probs.get)
    max_prob = normalized_probs[best_race]

    if max_prob >= threshold:
        predicted_labels[idx] = best_race
        filled_count += 1

df.loc[predicted_labels.keys(), 'patient_race'] = pd.Series(predicted_labels)

print(f'전체 결측치 수: {total_missing}')
print(f'채워진 결측치 수 (확률 ≥ {threshold}): {filled_count}')
print(f'채워진 비율: {round(filled_count / total_missing * 100, 2)}%')
print(f'남은 결측치 수: {df["patient_race"].isna().sum()}')

### 3-8. zip3 그룹 내 중복 피처 제거
- zip3 + N02xPM25xcommute_time 그룹 내에서 분산이 없는 컬럼 제거

In [ ]:
columns_to_check = [col for col in df.columns if col not in ['patient_zip3', 'N02xPM25xcommute_time']]
dropped_cols = []

for col in columns_to_check:
    df['check'] = df.groupby(['patient_zip3', 'N02xPM25xcommute_time'])[col].transform('nunique')
    if df['check'].max() == 1:
        dropped_cols.append(col)
        df = df.drop(col, axis=1)

df = df.drop('check', axis=1)
print(f'제거된 컬럼 수: {len(dropped_cols)}')
print('제거된 컬럼:', dropped_cols)

In [ ]:
# 불필요 컬럼 추가 제거
drop_list = ['Region', 'patient_zip3', 'desc_lower',
             'breast_cancer_diagnosis_code', 'tumor_site2']
drop_list = [c for c in drop_list if c in df.columns]

df = df.drop(columns=drop_list)
print(f'남은 컬럼 수: {len(df.columns)}')

## 4. 데이터 전처리

### 4-1. One-Hot Encoding (범주형 피처)

In [ ]:
categorical_cols = [
    col for col in df.columns
    if df[col].dtype == 'object' and col != 'tumor_site'
]
print('One-Hot Encoding 대상:', categorical_cols)

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype='int')

### 4-2. tumor_site 순서형 인코딩
- 의미론적 순서 반영: Central(4) > Overlapping(3) > Inner(2) > Outer(1) > Unspecified(0)

In [ ]:
tumor_site_order = {
    'Central': 4,
    'Overlapping': 3,
    'Inner': 2,
    'Outer': 1,
    'Unspecified': 0
}

df['tumor_site'] = df['tumor_site'].fillna('Unspecified')
df['tumor_site_encoded'] = df['tumor_site'].map(tumor_site_order)
df = df.drop(columns=['tumor_site'])

print('tumor_site_encoded 분포:')
print(df['tumor_site_encoded'].value_counts())

### 4-3. 나머지 범주형 컬럼 Label Encoding

In [ ]:
le = LabelEncoder()
for col in cat_cols:
    if col in df.columns:
        try:
            df[col] = le.fit_transform(df[col].astype(str)).astype('int')
        except Exception:
            continue

### 4-4. Train / Test 분리

In [ ]:
tdf = df[df[target].isna()].copy()
df = df[df[target].notna()].copy()

print(f'Train: {df.shape}, Test: {tdf.shape}')

## 5. 모델링 — 스태킹 앙상블

### 아키텍처
- **Level-1 Base Models**: CatBoost, RandomForest, XGBoost, LightGBM
- **Level-2 Meta Model**: Logistic Regression
- **검증 전략**: 10-Fold Stratified K-Fold

In [ ]:
# Base Models 정의
modela = CatBoostClassifier(
    iterations=500, silent=True,
    learning_rate=0.05, depth=10,
    eval_metric='AUC', random_seed=42
)

modelb = RandomForestClassifier(
    n_estimators=200, max_depth=None,
    min_samples_split=2, random_state=42, n_jobs=-1
)

modelc = XGBClassifier(
    learning_rate=0.1, max_depth=6,
    n_estimators=100, subsample=0.9,
    verbosity=0
)

modeld = LGBMClassifier(
    learning_rate=0.05, max_depth=-1,
    n_estimators=200, subsample=0.8,
    colsample_bytree=0.8, random_state=42,
    n_jobs=-1, verbose=-1
)

# Meta Model
meta_model = LogisticRegression()
meta_features = ['pred1', 'pred2', 'pred3', 'pred4']

In [ ]:
# 학습 피처 선택
drop_cols = ['patient_id', target, 'patient_state']
drop_cols = [c for c in drop_cols if c in df.columns]
cols = [c for c in df.columns if c not in drop_cols]

print(f'학습 피처 수: {len(cols)}')

In [ ]:
# 10-Fold Stratified K-Fold 스태킹
num_folds = 10
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

predictions_from_folds = pd.DataFrame()

for fold, (train_index, val_index) in enumerate(kf.split(df, df[target])):
    print(f'Fold {fold + 1}/{num_folds} 학습 중...', end=' ')

    dfx = df.iloc[train_index]
    efx = df.iloc[val_index].copy()

    # Level-1: Base model 예측
    efx['pred1'] = modela.fit(dfx[cols].values, dfx[target]).predict_proba(efx[cols].values)[:, 1]
    efx['pred2'] = modelb.fit(dfx[cols].values, dfx[target]).predict_proba(efx[cols].values)[:, 1]
    efx['pred3'] = modelc.fit(dfx[cols].values, dfx[target]).predict_proba(efx[cols].values)[:, 1]
    efx['pred4'] = modeld.fit(dfx[cols].values, dfx[target]).predict_proba(efx[cols].values)[:, 1]

    tdf_fold = tdf.copy()
    tdf_fold['pred1'] = modela.predict_proba(tdf[cols].values)[:, 1]
    tdf_fold['pred2'] = modelb.predict_proba(tdf[cols].values)[:, 1]
    tdf_fold['pred3'] = modelc.predict_proba(tdf[cols].values)[:, 1]
    tdf_fold['pred4'] = modeld.predict_proba(tdf[cols].values)[:, 1]

    # Level-2: Meta model 예측
    tdf_fold['pred'] = meta_model.fit(
        efx[meta_features], efx[target]
    ).predict_proba(tdf_fold[meta_features])[:, 1]

    predictions_from_folds = pd.concat([predictions_from_folds, tdf_fold], axis=0)
    print('완료')

print('\n모든 폴드 학습 완료!')

## 6. 피처 중요도 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

model_names = ['CatBoost', 'RandomForest', 'XGBoost', 'LightGBM']
models = [modela, modelb, modelc, modeld]

for ax, model, name in zip(axes.flatten(), models, model_names):
    fi = pd.DataFrame({'Feature': cols, 'Value': model.feature_importances_})
    fi = fi.sort_values('Value', ascending=False).head(15)
    sns.barplot(x='Value', y='Feature', data=fi, ax=ax)
    ax.set_title(f'{name} Top-15 Feature Importance')
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

## 7. 제출 파일 생성

In [ ]:
final_predictions = predictions_from_folds.groupby('patient_id').mean().reset_index()
final_predictions[target] = final_predictions['pred'].values

submission = final_predictions[ss.columns]
submission.to_csv('final_predictions.csv', index=False)

print('제출 파일 저장 완료: final_predictions.csv')
print(submission.describe())
submission.head()